In [1]:
# ==========================================
# ACC102 Mini Assignment - Data Preparation
# Track 4: Interactive Data Product
# ==========================================

import wrds
import pandas as pd

# 【封装为函数 Package entire workflow into one def】
def fetch_and_clean_qsr_data(username):
    print("正在连接 WRDS 数据库，请稍候...")
    db = wrds.Connection(wrds_username=username)
    
    # 1. 【Parameterised SQL 参数化查询】
    # 将公司代码和起始年份作为变量存储
    tickers = ('MCD', 'YUM', 'SBUX', 'SHAK')
    start_year = 2019
    
    # 2. 【f-string 注入 SQL 变量 】
    query = f"""
    SELECT tic, fyear, sale, ni, at, seq
    FROM comp.funda
    WHERE tic IN {tickers}
    AND fyear >= {start_year}
    AND indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C'
    """
    
    # 执行 SQL 查询获取数据
    df_raw = db.raw_sql(query)
    print("原始数据提取成功！")
    
    # 3. 【自动重命名技巧 builds rename dict automatically】
    raw_names = ['tic', 'fyear', 'sale', 'ni', 'at', 'seq']
    clean_names = ['Company', 'Year', 'Revenue', 'Net_Income', 'Total_Assets', 'Equity']
    
    # 使用 dict(zip()) 组合两个列表并重命名
    rename_map = dict(zip(raw_names, clean_names))
    df_clean = df_raw.rename(columns=rename_map)
    
    # 4. 【数据清洗 Data Cleaning】
    # 检查并删除缺失值，防止后续计算出错
    df_clean = df_clean.dropna()
    
    # 5. 将清洗好的数据导出为 CSV，供后续网页产品使用
    df_clean.to_csv('qsr_financials.csv', index=False)
    print("✅ 数据已成功清洗并保存为 qsr_financials.csv！")
    
    db.close()
    return df_clean

# 🚨 请在这里填入你自己的 WRDS 用户名并运行！
df_final = fetch_and_clean_qsr_data('jackiezh')

# 预览前5行数据
df_final.head()

正在连接 WRDS 数据库，请稍候...
Loading library list...
Done
原始数据提取成功！
✅ 数据已成功清洗并保存为 qsr_financials.csv！


,Company,Year,Revenue,Net_Income,Total_Assets,Equity
0,MCD,2019,21076.5,6025.4,47510.8,-8210.3
1,MCD,2020,19207.8,4730.5,52626.8,-7824.9
2,MCD,2021,23222.9,7545.2,53854.3,-4601.0
3,MCD,2022,23182.6,6177.4,50435.6,-6003.4
4,MCD,2023,25493.7,8468.8,56146.8,-4706.7
